### Problema 1: Detección de scrapers en una ventana de tráfico


Creamos la sesion de spark
Cargamos el dataset e imprimimos el schema

In [10]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MeliChallenge")
    .getOrCreate()
)

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../../data/session_requests.csv")
)

df.printSchema()

root
 |-- session_id: string (nullable = true)
 |-- request_time: timestamp (nullable = true)
 |-- path: string (nullable = true)
 |-- path_type: string (nullable = true)
 |-- d2id: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- user_agent: string (nullable = true)



![Descripción](images/image.png)

Visualizamos el dataframe

In [ ]:
# Vemos las primeras 10 filas del dataframe
df.show(10, truncate=False)

+-----------+-----------------------+--------------------+---------+------------------------------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------+
|session_id |request_time           |path                |path_type|d2id                                |ip_address    |user_agent                                                                                                                             |
+-----------+-----------------------+--------------------+---------+------------------------------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------+
|session_918|2026-06-15 14:00:01.947|/search?q=mesa      |search   |4504399e-5ac7-4fca-8755-7bc35e9f74be|104.161.135.39|Mozilla/5.0 (iPhone; CPU iPhone OS 17_4 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 

Empezamos con el analisis de los datos

In [12]:
# Vemos el largo del dataframe y la cantidad de valores distintos de cada columna

from pyspark.sql.functions import countDistinct
print(f"Largo dataframe: {df.count()}")
for column in df.columns:
    count = df.select(countDistinct(column)).collect()[0][0]
    print(f"{column}: {count}")

Largo dataframe: 186610
session_id: 1350
request_time: 185242
path: 68168
path_type: 2
d2id: 13773
ip_address: 2264
user_agent: 9


Algunos datos sobre esto:

- Largo dataframe (186,610) vs session_id (1,350):Es una interactividad altísima. Esto significa que los usuarios no entran y se van inmediatamente, sino que pasan tiempo navegando, buscando productos o recorriendo el sitio.

- path (68,168) vs path_type (2): El dataset está concentrado exclusivamente en el descubrimiento y compra

- d2id (13,773) vs session_id (1,350): Hay 10 veces más identificadores de dispositivo (d2id) que sesiones activas.

- ip_address (2,264) vs session_id (1,350): puede que predomine el tráfico en movimiento (móvil) donde los usuarios cambian de Wi-Fi a datos móviles o bien hay presencia de bots/scrapers que rotan IPs constantemente para evadir bloqueos mientras extraen información.



In [ ]:
# Vemos si hay valores nulos: El dataset no contiene valores nulos

from pyspark.sql.functions import col, sum

nulls = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

nulls.show()


+----------+------------+----+---------+----+----------+----------+
|session_id|request_time|path|path_type|d2id|ip_address|user_agent|
+----------+------------+----+---------+----+----------+----------+
|         0|           0|   0|        0|   0|         0|         0|
+----------+------------+----+---------+----+----------+----------+



In [ ]:
# Vemos si hay filas duplicadas: el dataset no contiene filas duplicadas

total_rows = df.count()
distinct_rows = df.distinct().count()

print(f"Total de filas: {total_rows}")
print(f"Filas distintas: {distinct_rows}")
print(f"Duplicados: {total_rows - distinct_rows}")



Total de filas: 186610
Filas distintas: 186610
Duplicados: 0


In [ ]:
from pyspark.sql.functions import min, max, date_format, round
# Vemos la ventana de tiempo que tiene el dataframe
df.select(
    date_format(min("request_time"), "yyyy-MM-dd HH:mm").alias("inicio"),
    date_format(max("request_time"), "yyyy-MM-dd HH:mm").alias("fin"),
    round(((max("request_time").cast("long") - min("request_time").cast("long"))/60/60),2).alias("Diferencia en horas")
).show()

+----------------+----------------+-------------------+
|          inicio|             fin|Diferencia en horas|
+----------------+----------------+-------------------+
|2026-06-15 14:00|2026-06-15 18:18|               4.31|
+----------------+----------------+-------------------+



In [ ]:
# Vemos que porcentaje es de busqueda(37%) y que porcentaje es de item(63%) para entender el comportamiento de los usuarios
# Se puede notar que pasan mas tiempo comparando items que haciendo una busqueda
total = df.count()
(
    df.groupBy("path_type")
    .count()
    .withColumn("percentage (%)", round(col("count") / total * 100,2))
    .show()
)

+---------+------+--------------+
|path_type| count|percentage (%)|
+---------+------+--------------+
|     item|118428|         63.46|
|   search| 68182|         36.54|
+---------+------+--------------+



In [ ]:
from pyspark.sql.functions import regexp_extract

# Ahora si veamos en busquedas, la cantidad de elementos distintos y la cantidad de cada uno de ellos

searches = (
    df
    .filter(df.path_type == "search")
    .withColumn(
        "search_query",
        regexp_extract("path", r"[?&]q=([^&]+)", 1)
    )
)

from pyspark.sql.functions import count, col

# 1. Agrupamos, contamos, ORDENAMOS de mayor a menor y tomamos los primeros 20
busquedas_con_conteo = (searches
                        .groupBy("search_query")
                        .agg(count("*").alias("total"))
                        .orderBy(col("total").desc())  # <-- El ordenamiento va aquí afuera
                        .collect())

# 2. Creamos la lista con el formato "elemento : cantidad"
lista_busquedas = [f"{row['search_query']} : {row['total']}" for row in busquedas_con_conteo]

print(lista_busquedas)
print("Cantidad de elementos distintos:", searches.select("search_query").distinct().count())

# Estos 25 productos tienen una cantidad de elementos cercana.


['parlante : 2811', 'auriculares : 2796', 'cargador : 2789', 'campera : 2785', 'sillon : 2769', 'notebook : 2767', 'bicicleta : 2764', 'silla : 2760', 'televisor : 2756', 'microondas : 2754', 'patineta : 2740', 'cortina : 2735', 'guitarra : 2731', 'cafetera : 2727', 'heladera : 2719', 'teclado : 2713', 'perfume : 2713', 'reloj : 2705', 'ventilador : 2701', 'celular : 2700', 'mouse : 2697', 'mesa : 2664', 'mochila : 2639', 'lampara : 2636', 'zapatillas : 2611']
Cantidad de elementos distintos: 25


In [18]:
from pyspark.sql.functions import regexp_extract, count, col

# Vemos el id de los item, buscando en path y como se repiten en las busquedas
# 1. Filtramos por "item" y extraemos el item_id
items = (
    df
    .filter(df.path_type == "item")
    .withColumn(
        "item_id",
        regexp_extract("path", r"/item/([^/?]+)", 1)
    )
)

# 2. Agrupamos por item_id, contamos, ORDENAMOS de mayor a menor y tomamos los primeros 20
items_con_conteo = (items
                    .groupBy("item_id")
                    .agg(count("*").alias("total"))
                    .orderBy(col("total").desc())
                    .take(50))

# 3. Creamos la lista con el formato "elemento : cantidad"
lista_items = [f"{row['item_id']} : {row['total']}" for row in items_con_conteo]

# 4. Imprimimos los resultados en consola
print(lista_items)
print("Cantidad de elementos distintos:", items.select("item_id").distinct().count())


# Podemos ver que no hay un "Producto Estrella" dominante
# Los 20 productos principales apenas suman unas 166 visitas en conjunto.

['MLA175177 : 11', 'MLA118567 : 10', 'MLA120070 : 9', 'MLA118544 : 9', 'MLA118255 : 9', 'MLA118546 : 9', 'MLA175940 : 9', 'MLA186831 : 8', 'MLA194052 : 8', 'MLA183370 : 8', 'MLA156899 : 8', 'MLA164982 : 8', 'MLA113497 : 8', 'MLA118385 : 8', 'MLA164060 : 8', 'MLA141409 : 8', 'MLA141625 : 7', 'MLA141730 : 7', 'MLA183422 : 7', 'MLA175154 : 7', 'MLA152793 : 7', 'MLA146340 : 7', 'MLA104121 : 7', 'MLA152845 : 7', 'MLA138452 : 7', 'MLA193361 : 7', 'MLA166176 : 7', 'MLA141469 : 7', 'MLA118518 : 7', 'MLA165972 : 7', 'MLA153253 : 7', 'MLA118336 : 7', 'MLA152676 : 7', 'MLA141845 : 7', 'MLA175101 : 7', 'MLA150469 : 7', 'MLA152691 : 7', 'MLA175520 : 7', 'MLA157199 : 7', 'MLA110053 : 7', 'MLA119527 : 7', 'MLA127773 : 7', 'MLA141585 : 7', 'MLA175279 : 7', 'MLA111428 : 7', 'MLA153793 : 7', 'MLA183310 : 7', 'MLA141771 : 7', 'MLA183607 : 7', 'MLA152979 : 7']
Cantidad de elementos distintos: 68143


In [ ]:
# La cantidad de ips diferentes por cada sesion

session_ips = (
    df
    .groupBy("session_id")
    .agg(
        countDistinct("ip_address").alias("unique_ips")
    )
    .orderBy("unique_ips", ascending=False)
)

session_ips.show(20)

## Aca ya notamos un comportamiento anomalo: Es en principio sospechoso que haya pocas sesiones con tal cantidad de ips.

+------------+----------+
|  session_id|unique_ips|
+------------+----------+
|session_1340|       138|
|session_1336|       137|
|session_1332|       126|
|session_1335|       112|
|session_1339|        88|
|session_1331|        74|
|session_1337|        74|
|session_1338|        70|
|session_1334|        54|
|session_1333|        51|
|session_1253|         1|
|session_1011|         1|
| session_564|         1|
| session_808|         1|
| session_339|         1|
|session_1322|         1|
| session_708|         1|
| session_237|         1|
| session_834|         1|
| session_885|         1|
+------------+----------+
only showing top 20 rows


In [ ]:
# La cantidad de sesiones diferentes por cada ip

ip_sessions = (
    df
    .groupBy("ip_address")
    .agg(
        countDistinct("session_id").alias("unique_sessions")
    )
    .orderBy("unique_sessions", ascending=False)
)

ip_sessions.show(20)

# Toda ip solo forma parte de una sesion.

+---------------+---------------+
|     ip_address|unique_sessions|
+---------------+---------------+
|  137.98.245.82|              1|
|121.229.250.133|              1|
|    82.87.82.19|              1|
| 79.248.223.134|              1|
| 132.177.198.81|              1|
| 179.232.70.224|              1|
|  113.95.222.41|              1|
|196.205.170.127|              1|
|  218.53.55.138|              1|
|  98.206.44.232|              1|
|  187.84.245.33|              1|
|  21.253.202.35|              1|
| 172.145.133.76|              1|
| 43.221.169.221|              1|
| 139.177.203.57|              1|
| 201.11.175.236|              1|
|  65.240.99.253|              1|
| 73.124.146.240|              1|
| 167.197.34.245|              1|
|  76.35.186.222|              1|
+---------------+---------------+
only showing top 20 rows


In [ ]:
# La cantidad de dispositivos por cada sesion

session_devices = (
    df
    .groupBy("session_id")
    .agg(
        countDistinct("d2id").alias("unique_devices")
    )
    .orderBy("unique_devices", ascending=False)
)

session_devices.show(10)
session_devices.orderBy("unique_devices",asc=False).show(10)
# Aca podriamos ver los percentiles
from pyspark.sql.functions import expr, percentile_approx

# Calculamos los percentiles sobre la columna 'unique_devices'
session_devices.select(
    percentile_approx("unique_devices", 0.25).alias("p25"),
    percentile_approx("unique_devices", 0.50).alias("p50_mediana"),
    percentile_approx("unique_devices", 0.75).alias("p75"),
    percentile_approx("unique_devices", 0.90).alias("p90"),
    percentile_approx("unique_devices", 0.99).alias("p99")
).show()

# Aca notamos tambien que la cantidad de dispositivos por sesion tambien tiene un comportamiento extraño

+------------+--------------+
|  session_id|unique_devices|
+------------+--------------+
|session_1302|          1167|
|session_1305|          1128|
|session_1309|          1097|
|session_1303|          1066|
|session_1306|           909|
|session_1311|           904|
|session_1304|           760|
|session_1314|           684|
|session_1315|           677|
|session_1312|           660|
+------------+--------------+
only showing top 10 rows
+------------+--------------+
|  session_id|unique_devices|
+------------+--------------+
| session_244|             1|
| session_463|             1|
| session_414|             1|
|session_1253|             1|
|  session_10|             1|
|session_1011|             1|
| session_211|             1|
| session_708|             1|
| session_689|             1|
|session_1083|             1|
+------------+--------------+
only showing top 10 rows
+---+-----------+---+---+---+
|p25|p50_mediana|p75|p90|p99|
+---+-----------+---+---+---+
|  1|          1|  1

In [ ]:
# Vemos la cantidad de requests que hay por sesion, y hacemos un summary para detectar si hay comportamientos extraños.

from pyspark.sql.functions import count


requests_per_session = (
    df.groupBy("session_id")
    .agg(
        count("*").alias("request_count")
    ).orderBy("request_count", ascending=False)
    
)
requests_per_session.show(10)
print("\n")
requests_per_session.select("request_count").summary().show()

NameError: name 'df' is not defined

Vamos a construir el dataset de features ->Agrupandolo por session id

In [23]:
# ============================================================
# 1. COMPORTAMIENTO GENERAL DE CADA SESIÓN
# ============================================================

from pyspark.sql.functions import (
    count,
    countDistinct,
    sum,
    when,
    min,
    max,
    col,
    unix_micros,
    lag,
    avg,
    expr,
    stddev
)
from pyspark.sql.window import Window


session_behavior = (
    df
    .groupBy("session_id")
    .agg(
        # Cantidad de requests
        count("*").alias("request_count"),

        # Duración
        min("request_time").alias("session_start"),
        max("request_time").alias("session_end"),

        # Tipo de request
        sum(
            when(col("path_type") == "search", 1).otherwise(0)
        ).alias("search_requests"),

        sum(
            when(col("path_type") == "item", 1).otherwise(0)
        ).alias("item_requests"),

        # Diversidad
        countDistinct("path").alias("unique_paths"),
        countDistinct("ip_address").alias("unique_ips"),
        countDistinct("d2id").alias("unique_devices"),
        countDistinct("user_agent").alias("unique_user_agents")
    )
    .withColumn(
        "duration_seconds",
        (
            unix_micros("session_end") -
            unix_micros("session_start")
        ) / 1_000_000
    )
    .withColumn(
        "requests_per_second",
        when(
            col("duration_seconds") > 0,
            col("request_count") / col("duration_seconds")
        ).otherwise(0)
    )
    .withColumn(
        "requests_per_minute",
        col("requests_per_second") * 60
    )
)


# ============================================================
# 2. INTERVALOS ENTRE REQUESTS
# ============================================================

window = (
    Window
    .partitionBy("session_id")
    .orderBy("request_time")
)

df_intervals = (
    df
    .withColumn(
        "previous_request_time",
        lag("request_time").over(window)
    )
    .withColumn(
        "interval_seconds",
        (
            unix_micros("request_time") -
            unix_micros("previous_request_time")
        ) / 1_000_000
    )
)


# ============================================================
# 3. COMPORTAMIENTO TEMPORAL DE CADA SESIÓN
# ============================================================

interval_features = (
    df_intervals
    .groupBy("session_id")
    .agg(
        min("interval_seconds").alias("min_interval_seconds"),

        avg("interval_seconds").alias(
            "avg_interval_seconds"
        ),

        expr(
            "percentile_approx(interval_seconds, 0.50)"
        ).alias(
            "median_interval_seconds"
        ),

        expr(
            "percentile_approx(interval_seconds, 0.95)"
        ).alias(
            "p95_interval_seconds"
        ),

        stddev("interval_seconds").alias(
            "interval_stddev_seconds"
        )
    )
)


# ============================================================
# 4. INFORMACIÓN DE LOS DISPOSITIVOS
# ============================================================

device_features = (
    df
    .groupBy("d2id")
    .agg(
        count("*").alias("device_request_count"),

        countDistinct("session_id").alias(
            "device_session_count"
        ),

        countDistinct("ip_address").alias(
            "device_unique_ips"
        ),

        countDistinct("user_agent").alias(
            "device_unique_user_agents"
        )
    )
)


# Relacionamos cada sesión con sus dispositivos
df_device = (
    df
    .select("session_id", "d2id")
    .dropDuplicates()
    .join(
        device_features,
        on="d2id",
        how="left"
    )
)


# Llevamos la información del dispositivo al nivel sesión
session_device_features = (
    df_device
    .groupBy("session_id")
    .agg(
        max("device_request_count").alias(
            "device_request_count"
        ),

        max("device_session_count").alias(
            "device_session_count"
        ),

        max("device_unique_ips").alias(
            "device_unique_ips"
        ),

        max("device_unique_user_agents").alias(
            "device_unique_user_agents"
        )
    )
)


# ============================================================
# 5. CONSTRUIR DATASET FINAL DE FEATURES
# ============================================================

features = (
    session_behavior

    # Agregamos comportamiento temporal
    .join(
        interval_features,
        on="session_id",
        how="left"
    )

    # Agregamos contexto de dispositivos
    .join(
        session_device_features,
        on="session_id",
        how="left"
    )
)


# ============================================================
# 6. COEFICIENTE DE VARIACIÓN DE LOS INTERVALOS
# ============================================================

features = (
    features
    .withColumn(
        "interval_cv",
        when(
            col("avg_interval_seconds") > 0,
            col("interval_stddev_seconds") /
            col("avg_interval_seconds")
        ).otherwise(0)
    )
)


# ============================================================
# 7. VERIFICACIÓN
# ============================================================

features.printSchema()
features.show(5, truncate=False)

root
 |-- session_id: string (nullable = true)
 |-- request_count: long (nullable = false)
 |-- session_start: timestamp (nullable = true)
 |-- session_end: timestamp (nullable = true)
 |-- search_requests: long (nullable = true)
 |-- item_requests: long (nullable = true)
 |-- unique_paths: long (nullable = false)
 |-- unique_ips: long (nullable = false)
 |-- unique_devices: long (nullable = false)
 |-- unique_user_agents: long (nullable = false)
 |-- duration_seconds: double (nullable = true)
 |-- requests_per_second: double (nullable = true)
 |-- requests_per_minute: double (nullable = true)
 |-- min_interval_seconds: double (nullable = true)
 |-- avg_interval_seconds: double (nullable = true)
 |-- median_interval_seconds: double (nullable = true)
 |-- p95_interval_seconds: double (nullable = true)
 |-- interval_stddev_seconds: double (nullable = true)
 |-- device_request_count: long (nullable = true)
 |-- device_session_count: long (nullable = true)
 |-- device_unique_ips: long (nul

In [ ]:
# Imprimimos la cantidad de sesiones, features y el schema
print(f"Cantidad de sesiones: {features.count()}")
print(f"Cantidad de features: {len(features.columns)}")
features.printSchema()

root
 |-- session_id: string (nullable = true)
 |-- request_count: long (nullable = false)
 |-- session_start: timestamp (nullable = true)
 |-- session_end: timestamp (nullable = true)
 |-- search_requests: long (nullable = true)
 |-- item_requests: long (nullable = true)
 |-- unique_paths: long (nullable = false)
 |-- unique_ips: long (nullable = false)
 |-- unique_devices: long (nullable = false)
 |-- unique_user_agents: long (nullable = false)
 |-- duration_seconds: double (nullable = true)
 |-- requests_per_second: double (nullable = true)
 |-- requests_per_minute: double (nullable = true)
 |-- min_interval_seconds: double (nullable = true)
 |-- avg_interval_seconds: double (nullable = true)
 |-- median_interval_seconds: double (nullable = true)
 |-- p95_interval_seconds: double (nullable = true)
 |-- interval_stddev_seconds: double (nullable = true)
 |-- device_request_count: long (nullable = true)
 |-- device_session_count: long (nullable = true)
 |-- device_unique_ips: long (nul

In [27]:
# Columnas numéricas que nos interesan para analizar comportamiento
numeric_features = [
    "session_id",
    "request_count",
    "session_start",
    "session_end",
    "search_requests",
    "item_requests",
    "unique_paths",
    "unique_ips",
    "unique_devices",
    "unique_user_agents",
    "duration_seconds",
    "requests_per_second",
    "requests_per_minute",
    "min_interval_seconds",
    "avg_interval_seconds",
    "median_interval_seconds",
    "p95_interval_seconds",
    "interval_stddev_seconds",
    "interval_cv",
    "device_request_count",
    "device_session_count",
    "device_unique_ips",
    "device_unique_user_agents"
]

features.select(numeric_features).summary().show()

+-------+-----------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+-------------------+-------------------+--------------------+--------------------+-----------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+-----------------+-------------------------+
|summary| session_id|     request_count|   search_requests|    item_requests|      unique_paths|       unique_ips|    unique_devices|unique_user_agents|  duration_seconds|requests_per_second|requests_per_minute|min_interval_seconds|avg_interval_seconds|median_interval_seconds|p95_interval_seconds|interval_stddev_seconds|         interval_cv|device_request_count|device_session_count|device_unique_ips|device_unique_user_agents|
+-------+-----------+------------------+------------------+-----------------+------------------+-----------------+------------------+-------

Vemos que hay columnas que no tienen importancia como
device_session_count	
device_unique_user_agents
tienen un unico valor   

Luego vemos columnas de:
- Muy relevantes:
    requests_per_second # Podriamos borrarla porque tenemos a requests_per_minute
    requests_per_minute
    unique_devices
    median_interval_seconds
    interval_cv = interval_stddev_seconds / avg_interval_seconds : Por eso después vamos a combinarlo con: requests_per_minut median_interval unique_devices unique_ips para ver si es relevante
- Relevantes:
    request_count
    unique_ips
    unique_paths
    item_requests
    avg_interval_seconds # Pero está bastante relacionado con requests_per_minute, así que después podemos analizar si necesitamos ambos.
    interval_stddev_seconds 
- Medianamente relevantes:
    unique_user_agents
    device_request_count
    device_unique_ips
    min_interval_seconds: Es interesante, pero tiene un problemaUna sesión humana puede tener un request accidentalmente muy rápido.Por eso min_interval por sí solo no es una buena regla.
    p95_interval_seconds
- Poco relevantes:
    search_requests
    duration_seconds
    


In [28]:
features = features.drop(
    "device_session_count",
    "device_unique_user_agents",
    "requests_per_second"
)

features.printSchema()

root
 |-- session_id: string (nullable = true)
 |-- request_count: long (nullable = false)
 |-- session_start: timestamp (nullable = true)
 |-- session_end: timestamp (nullable = true)
 |-- search_requests: long (nullable = true)
 |-- item_requests: long (nullable = true)
 |-- unique_paths: long (nullable = false)
 |-- unique_ips: long (nullable = false)
 |-- unique_devices: long (nullable = false)
 |-- unique_user_agents: long (nullable = false)
 |-- duration_seconds: double (nullable = true)
 |-- requests_per_minute: double (nullable = true)
 |-- min_interval_seconds: double (nullable = true)
 |-- avg_interval_seconds: double (nullable = true)
 |-- median_interval_seconds: double (nullable = true)
 |-- p95_interval_seconds: double (nullable = true)
 |-- interval_stddev_seconds: double (nullable = true)
 |-- device_request_count: long (nullable = true)
 |-- device_unique_ips: long (nullable = true)
 |-- interval_cv: double (nullable = true)



### CODIGO

In [29]:
from pyspark.sql.functions import expr

features.select(
    expr("percentile_approx(request_count, array(0.01, 0.05, 0.50, 0.95, 0.99))").alias("request_count"),
    expr("percentile_approx(requests_per_minute, array(0.01, 0.05, 0.50, 0.95, 0.99))").alias("requests_per_minute"),
    expr("percentile_approx(unique_devices, array(0.01, 0.05, 0.50, 0.95, 0.99))").alias("unique_devices"),
    expr("percentile_approx(unique_ips, array(0.01, 0.05, 0.50, 0.95, 0.99))").alias("unique_ips"),
    expr("percentile_approx(unique_paths, array(0.01, 0.05, 0.50, 0.95, 0.99))").alias("unique_paths"),
    expr("percentile_approx(median_interval_seconds, array(0.01, 0.05, 0.50, 0.95, 0.99))").alias("median_interval_seconds"),
    expr("percentile_approx(interval_cv, array(0.01, 0.05, 0.50, 0.95, 0.99))").alias("interval_cv")
).show(truncate=False)

+-----------------------+-------------------------------------------------------------------------------------------------+-----------------+---------------+----------------------+------------------------------------+-----------------------------------------------------------------------------------------------------+
|request_count          |requests_per_minute                                                                              |unique_devices   |unique_ips     |unique_paths          |median_interval_seconds             |interval_cv                                                                                          |
+-----------------------+-------------------------------------------------------------------------------------------------+-----------------+---------------+----------------------+------------------------------------+-----------------------------------------------------------------------------------------------------+
|[60, 66, 124, 185, 551]|[1.161142522399

In [30]:
thresholds = features.select(
    expr("percentile_approx(request_count, 0.99)").alias("p99_request_count"),
    expr("percentile_approx(requests_per_minute, 0.99)").alias("p99_requests_per_minute"),
    expr("percentile_approx(unique_devices, 0.99)").alias("p99_unique_devices"),
    expr("percentile_approx(unique_paths, 0.99)").alias("p99_unique_paths"),
    expr("percentile_approx(median_interval_seconds, 0.05)").alias("p05_median_interval")
).first()

print(thresholds)

Row(p99_request_count=551, p99_requests_per_minute=25.242119948737084, p99_unique_devices=475, p99_unique_paths=520, p05_median_interval=12.158)


In [31]:
features = (
    features

    # Regla 1: cantidad extrema de requests
    .withColumn(
        "rule_high_request_count",
        when(
            col("request_count") > thresholds["p99_request_count"],
            1
        ).otherwise(0)
    )

    # Regla 2: velocidad extrema
    .withColumn(
        "rule_high_request_rate",
        when(
            col("requests_per_minute") > thresholds["p99_requests_per_minute"],
            1
        ).otherwise(0)
    )

    # Regla 3: cantidad extrema de dispositivos
    .withColumn(
        "rule_high_unique_devices",
        when(
            col("unique_devices") > thresholds["p99_unique_devices"],
            1
        ).otherwise(0)
    )

    # Regla 4: diversidad extrema de paths
    .withColumn(
        "rule_high_unique_paths",
        when(
            col("unique_paths") > thresholds["p99_unique_paths"],
            1
        ).otherwise(0)
    )

    # Regla 5: cadencia muy rápida
    .withColumn(
        "rule_low_median_interval",
        when(
            col("median_interval_seconds") < thresholds["p05_median_interval"],
            1
        ).otherwise(0)
    )
)

In [32]:
features.select(
    "rule_high_request_count",
    "rule_high_request_rate",
    "rule_high_unique_devices",
    "rule_high_unique_paths",
    "rule_low_median_interval"
).groupBy(
    "rule_high_request_count",
    "rule_high_request_rate",
    "rule_high_unique_devices",
    "rule_high_unique_paths",
    "rule_low_median_interval"
).count().orderBy("count", ascending=False).show()

+-----------------------+----------------------+------------------------+----------------------+------------------------+-----+
|rule_high_request_count|rule_high_request_rate|rule_high_unique_devices|rule_high_unique_paths|rule_low_median_interval|count|
+-----------------------+----------------------+------------------------+----------------------+------------------------+-----+
|                      0|                     0|                       0|                     0|                       0| 1283|
|                      0|                     0|                       0|                     0|                       1|   50|
|                      1|                     1|                       1|                     1|                       1|    9|
|                      0|                     1|                       0|                     0|                       1|    2|
|                      0|                     1|                       1|                     0|        

In [33]:
from pyspark.sql.functions import col, when

features = (
    features
    .withColumn(
        "anomaly_score",
        col("rule_high_request_count") +
        col("rule_high_request_rate") +
        col("rule_high_unique_devices") +
        col("rule_high_unique_paths") +
        col("rule_low_median_interval")
    )
    .withColumn(
        "strong_signal",
        (
            col("rule_high_request_count") +
            col("rule_high_request_rate") +
            col("rule_high_unique_devices") +
            col("rule_high_unique_paths")
        )
    )
    .withColumn(
        "is_scraping",
        when(
            (col("anomaly_score") >= 2) &
            (col("strong_signal") >= 1),
            1
        ).otherwise(0)
    )
)

In [34]:
features.groupBy("is_scraping").count().show()

+-----------+-----+
|is_scraping|count|
+-----------+-----+
|          1|   17|
|          0| 1333|
+-----------+-----+



In [35]:
from pyspark.sql.functions import col, round

# 1. Obtenemos el total de registros del DataFrame
total_registros = features.count()

# 2. Agrupamos, contamos y calculamos el porcentaje en el mismo flujo
(
    features.groupBy("is_scraping")
    .count()
    .withColumn("porcentaje (%)", round((col("count") / total_registros) * 100, 2))
    .show()
)


+-----------+-----+--------------+
|is_scraping|count|porcentaje (%)|
+-----------+-----+--------------+
|          1|   17|          1.26|
|          0| 1333|         98.74|
+-----------+-----+--------------+



In [36]:
features.filter(
    col("is_scraping") == 1
).select(
    "session_id",
    "request_count",
    "requests_per_minute",
    "unique_devices",
    "unique_ips",
    "unique_paths",
    "item_requests",
    "median_interval_seconds",
    "interval_cv",
    "anomaly_score"
).orderBy(
    col("anomaly_score").desc()
).show(100, truncate=False)

+------------+-------------+-------------------+--------------+----------+------------+-------------+-----------------------+--------------------+-------------+
|session_id  |request_count|requests_per_minute|unique_devices|unique_ips|unique_paths|item_requests|median_interval_seconds|interval_cv         |anomaly_score|
+------------+-------------+-------------------+--------------+----------+------------+-------------+-----------------------+--------------------+-------------+
|session_1305|1128         |29.200330315793288 |1128          |1         |1108        |1089         |2.054                  |0.07518425553218114 |5            |
|session_1306|909          |45.26092249627183  |909           |1         |886         |865          |1.328                  |0.05299401424214056 |5            |
|session_1307|616          |53.66921121070829  |616           |1         |609         |597          |1.115                  |0.09606054016576537 |5            |
|session_1311|904          |44.289

In [37]:
p75_request_count = features.select(
    expr("percentile_approx(request_count, 0.75)")
).first()[0]

p75_requests_per_minute = features.select(
    expr("percentile_approx(requests_per_minute, 0.75)")
).first()[0]

In [38]:
features = features.withColumn(
    "rule_fast_high_activity",
    when(
        (col("median_interval_seconds") < thresholds["p05_median_interval"]) &
        (
            (col("request_count") > p75_request_count) |
            (col("requests_per_minute") > p75_requests_per_minute)
        ),
        1
    ).otherwise(0)
)

In [39]:
print("P75 request_count:", p75_request_count)
print("P75 requests_per_minute:", p75_requests_per_minute)
print("P05 median_interval:", thresholds["p05_median_interval"])

P75 request_count: 157
P75 requests_per_minute: 2.431360356599519
P05 median_interval: 12.158


In [40]:
features.groupBy(
    "rule_fast_high_activity"
).count().show()

+-----------------------+-----+
|rule_fast_high_activity|count|
+-----------------------+-----+
|                      1|   67|
|                      0| 1283|
+-----------------------+-----+



In [41]:
features.filter(
    col("rule_fast_high_activity") == 1
).select(
    "session_id",
    "request_count",
    "requests_per_minute",
    "unique_devices",
    "unique_paths",
    "item_requests",
    "median_interval_seconds",
    "interval_cv"
).orderBy(
    col("requests_per_minute").desc()
).show(100, truncate=False)

+------------+-------------+-------------------+--------------+------------+-------------+-----------------------+--------------------+
|session_id  |request_count|requests_per_minute|unique_devices|unique_paths|item_requests|median_interval_seconds|interval_cv         |
+------------+-------------+-------------------+--------------+------------+-------------+-----------------------+--------------------+
|session_1307|616          |53.66921121070829  |616           |609         |597          |1.115                  |0.09606054016576537 |
|session_1310|480          |52.48942010126084  |480           |476         |461          |1.143                  |0.06596275022815531 |
|session_1309|1097         |47.64817663559859  |1097          |1080        |1060         |1.26                   |0.06328699746733055 |
|session_1306|909          |45.26092249627183  |909           |886         |865          |1.328                  |0.05299401424214056 |
|session_1311|904          |44.28995220702613  |

In [42]:
features = features.withColumn(
    "is_scraping",
    when(
        (col("anomaly_score") >= 2) |
        (col("rule_fast_high_activity") == 1),
        1
    ).otherwise(0)
)

In [43]:
features = features.withColumn(
    "is_scraping",
    when(
        col("rule_fast_high_activity") == 1,
        1
    ).otherwise(0)
)

In [44]:
features.groupBy(
    "is_scraping"
).count().show()

+-----------+-----+
|is_scraping|count|
+-----------+-----+
|          1|   67|
|          0| 1283|
+-----------+-----+



In [45]:
features.filter(
    col("is_scraping") == 1
).select(
    "session_id",
    "request_count",
    "requests_per_minute",
    "unique_devices",
    "unique_ips",
    "unique_paths",
    "item_requests",
    "median_interval_seconds",
    "interval_cv",
    "anomaly_score",
    "rule_fast_high_activity"
).orderBy(
    col("requests_per_minute").desc()
).show(100, truncate=False)

+------------+-------------+-------------------+--------------+----------+------------+-------------+-----------------------+--------------------+-------------+-----------------------+
|session_id  |request_count|requests_per_minute|unique_devices|unique_ips|unique_paths|item_requests|median_interval_seconds|interval_cv         |anomaly_score|rule_fast_high_activity|
+------------+-------------+-------------------+--------------+----------+------------+-------------+-----------------------+--------------------+-------------+-----------------------+
|session_1307|616          |53.66921121070829  |616           |1         |609         |597          |1.115                  |0.09606054016576537 |5            |1                      |
|session_1310|480          |52.48942010126084  |480           |1         |476         |461          |1.143                  |0.06596275022815531 |3            |1                      |
|session_1309|1097         |47.64817663559859  |1097          |1         |1

In [46]:
features.select(
    (sum(col("is_scraping")) / count("*") * 100).alias("scraping_percentage")
).show()

+-------------------+
|scraping_percentage|
+-------------------+
|  4.962962962962963|
+-------------------+



In [47]:
features.filter(
    col("is_scraping") == 1
).select(
    "session_id",
    "request_count",
    "requests_per_minute",
    "unique_devices",
    "unique_paths",
    "item_requests",
    "median_interval_seconds",
    "interval_cv"
).orderBy(
    col("requests_per_minute").desc()
).show(100, truncate=False)

+------------+-------------+-------------------+--------------+------------+-------------+-----------------------+--------------------+
|session_id  |request_count|requests_per_minute|unique_devices|unique_paths|item_requests|median_interval_seconds|interval_cv         |
+------------+-------------+-------------------+--------------+------------+-------------+-----------------------+--------------------+
|session_1307|616          |53.66921121070829  |616           |609         |597          |1.115                  |0.09606054016576537 |
|session_1310|480          |52.48942010126084  |480           |476         |461          |1.143                  |0.06596275022815531 |
|session_1309|1097         |47.64817663559859  |1097          |1080        |1060         |1.26                   |0.06328699746733055 |
|session_1306|909          |45.26092249627183  |909           |886         |865          |1.328                  |0.05299401424214056 |
|session_1311|904          |44.28995220702613  |

In [48]:
final_features = features.select(
    "session_id",
    "request_count",
    "requests_per_minute",
    "unique_devices",
    "unique_ips",
    "unique_paths",
    "item_requests",
    "search_requests",
    "duration_seconds",
    "median_interval_seconds",
    "avg_interval_seconds",
    "interval_cv",
    "is_scraping"
)

In [50]:
final_features.printSchema()

root
 |-- session_id: string (nullable = true)
 |-- request_count: long (nullable = false)
 |-- requests_per_minute: double (nullable = true)
 |-- unique_devices: long (nullable = false)
 |-- unique_ips: long (nullable = false)
 |-- unique_paths: long (nullable = false)
 |-- item_requests: long (nullable = true)
 |-- search_requests: long (nullable = true)
 |-- duration_seconds: double (nullable = true)
 |-- median_interval_seconds: double (nullable = true)
 |-- avg_interval_seconds: double (nullable = true)
 |-- interval_cv: double (nullable = true)
 |-- is_scraping: integer (nullable = false)



In [51]:
final_features.groupBy(
    "is_scraping"
).count().show()

+-----------+-----+
|is_scraping|count|
+-----------+-----+
|          1|   67|
|          0| 1283|
+-----------+-----+



In [52]:
sessions_for_llm = [
    "session_1307",
    "session_1331",
    "session_995"
]

sessions_df = (
    final_features
    .filter(col("session_id").isin(sessions_for_llm))
    .orderBy("session_id")
)

sessions_df.show(truncate=False)

+------------+-------------+-------------------+--------------+----------+------------+-------------+---------------+----------------+-----------------------+--------------------+-------------------+-----------+
|session_id  |request_count|requests_per_minute|unique_devices|unique_ips|unique_paths|item_requests|search_requests|duration_seconds|median_interval_seconds|avg_interval_seconds|interval_cv        |is_scraping|
+------------+-------------+-------------------+--------------+----------+------------+-------------+---------------+----------------+-----------------------+--------------------+-------------------+-----------+
|session_1307|616          |53.66921121070829  |616           |1         |609         |597          |19             |688.663         |1.115                  |1.1197772357723577  |0.09606054016576537|1          |
|session_1331|293          |21.67194493994603  |1             |74        |278         |261          |32             |811.187         |2.795             

In [53]:
sessions_data = [
    row.asDict()
    for row in sessions_df.collect()
]

sessions_data

[{'session_id': 'session_1307',
  'request_count': 616,
  'requests_per_minute': 53.66921121070829,
  'unique_devices': 616,
  'unique_ips': 1,
  'unique_paths': 609,
  'item_requests': 597,
  'search_requests': 19,
  'duration_seconds': 688.663,
  'median_interval_seconds': 1.115,
  'avg_interval_seconds': 1.1197772357723577,
  'interval_cv': 0.09606054016576537,
  'is_scraping': 1},
 {'session_id': 'session_1331',
  'request_count': 293,
  'requests_per_minute': 21.67194493994603,
  'unique_devices': 1,
  'unique_ips': 74,
  'unique_paths': 278,
  'item_requests': 261,
  'search_requests': 32,
  'duration_seconds': 811.187,
  'median_interval_seconds': 2.795,
  'avg_interval_seconds': 2.7780376712328767,
  'interval_cv': 0.2698517820231591,
  'is_scraping': 1},
 {'session_id': 'session_995',
  'request_count': 91,
  'requests_per_minute': 4.46080982489279,
  'unique_devices': 1,
  'unique_ips': 1,
  'unique_paths': 72,
  'item_requests': 52,
  'search_requests': 39,
  'duration_secon

FALTA PONER ESTADISTICAS COMUNES AL LLM, PARA QUE TENGA CONTEXTO TAMBIEN DE ESO

In [57]:
numeric_features_without_irrelevant = [
    feature for feature in numeric_features
    if feature not in [
        "device_session_count",
        "device_unique_user_agents",
        "requests_per_second"
    ]
]

summary_df = features.select(numeric_features_without_irrelevant).summary()

summary_df.show()

+-------+-----------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+-------------------+--------------------+--------------------+-----------------------+--------------------+-----------------------+--------------------+--------------------+-----------------+
|summary| session_id|     request_count|   search_requests|    item_requests|      unique_paths|       unique_ips|    unique_devices|unique_user_agents|  duration_seconds|requests_per_minute|min_interval_seconds|avg_interval_seconds|median_interval_seconds|p95_interval_seconds|interval_stddev_seconds|         interval_cv|device_request_count|device_unique_ips|
+-------+-----------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+-------------------+--------------------+--------------------+-----------------------+-----------------

In [58]:
summary_df = (
    features
    .select(numeric_features_without_irrelevant)
    .summary()
)

summary_df.show()

+-------+-----------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+-------------------+--------------------+--------------------+-----------------------+--------------------+-----------------------+--------------------+--------------------+-----------------+
|summary| session_id|     request_count|   search_requests|    item_requests|      unique_paths|       unique_ips|    unique_devices|unique_user_agents|  duration_seconds|requests_per_minute|min_interval_seconds|avg_interval_seconds|median_interval_seconds|p95_interval_seconds|interval_stddev_seconds|         interval_cv|device_request_count|device_unique_ips|
+-------+-----------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+-------------------+--------------------+--------------------+-----------------------+-----------------

In [59]:
summary_context = {}

for row in summary_df.collect():

    summary_type = row["summary"]

    for column in summary_df.columns:

        if column == "summary":
            continue

        if column not in summary_context:
            summary_context[column] = {}

        summary_context[column][summary_type] = row[column]

In [60]:
# EJEMPLO ELIGAMOS UNA SESION
session = (
    features
    .filter(col("session_id") == "session_1307")
    .first()
    .asDict()
)

In [ ]:
import json
from google import genai

# ==========================================
# 1. ELIGE AQUÍ TUS 3 SESIONES MANUALMENTE
# ==========================================
# Reemplaza estos nombres por los IDs exactos de tu DataFrame
mis_sesiones_elegidas = ["session_1307", "session_1042", "session_1115"]

print(f"Sesiones seleccionadas manualmente para el reporte: {mis_sesiones_elegidas}")

# ==========================================
# 2. CONFIGURAR EL CLIENTE DE GEMINI
# ==========================================
import os
from dotenv import load_dotenv
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")  # se carga desde el archivo .env, nunca hardcodear la key
client = genai.Client(api_key=GEMINI_API_KEY)

# Diccionario para guardar el texto de los 3 reportes
reportes_generados = {}

# ==========================================
# 3. BUCLE PARA GENERAR LOS 3 REPORTES ELEGIDOS
# ==========================================
for s_id in mis_sesiones_elegidas:
    print(f"\nProcesando reporte para la sesión: {s_id}...")
    
    # Extraemos los datos de la sesión actual como un diccionario de Python
    session_data = (
        features
        .filter(col("session_id") == s_id)
        .first()
        .asDict()
    )
    
    # Construimos el prompt dinámico inyectando la sesión correspondiente
    prompt = f"""
Estamos analizando sesiones HTTP para detectar comportamientos
anómalos potencialmente asociados con scraping.

Una etapa previa del sistema utiliza reglas estadísticas para
identificar sesiones sospechosas. Tu tarea NO es decidir si una
sesión es scraping. Tu tarea es generar un reporte explicativo
basado exclusivamente en los datos proporcionados.

CONTEXTO ESTADÍSTICO DE TODAS LAS SESIONES

{json.dumps(summary_context, indent=2)}

SESIÓN ANALIZADA

{json.dumps(session_data, indent=2, default=str)}

Analiza la sesión comparando sus métricas con la distribución
estadística de las demás sesiones.

El reporte debe cumplir estrictamente con los siguientes puntos:
1. Resumir el comportamiento de la sesión.
2. Identificar las métricas que presentan mayor desviación respecto de la población.
3. Comparar los valores de la sesión con estadísticas como media, mediana, percentiles y máximos cuando sea relevante.
4. Explicar conjuntamente volumen, frecuencia, diversidad de dispositivos/paths y comportamiento temporal.
5. Explicar por qué estas características justifican que la sesión haya sido marcada como anómala por el sistema.
6. No inventar información que no esté presente en los datos.

REGLAS CRÍTICAS DE ESTILO PARA COMPILACIÓN EN PDF:
7. Todo el texto que deba resaltar debe usar ÚNICAMENTE las etiquetas HTML <b>texto</b>.
8. Está TERMINANTEMENTE PROHIBIDO usar caracteres de Markdown como asteriscos (*) o acentos graves (`). No los uses bajo ninguna circunstancia. Si necesitas envolver el nombre de una variable o métrica, usa comillas normales (ejemplo: "request_count") o ponlo en negrita (ejemplo: <b>request_count</b>).
9. Estructura de Listas para PDF: En lugar de usar asteriscos para las viñetas, inicia la línea directamente con una tabulación o usa el carácter especial de viñeta plano o numeración manual (ejemplo: "1. ", "   - " o "   • "). Para generar sub-elementos, utiliza saltos de línea y espacios de tabulación tradicionales.
10. Todos los títulos principales, títulos secundarios y elementos que vayan luego de un "•" o "-" DEBEN ir envueltos en negrita HTML (ejemplo: <b>• Título:</b>).
11. No utilices el carácter "•" en los títulos principales o secundarios de sección. Deja siempre una línea en blanco (doble salto de línea) entre los diferentes títulos secundarios para que el PDF no quede amontonado.

SISTEMA DE PENALIZACIÓN DE FORMATO:
- SI USAS UNA SOLA COMA INVERTIDA (`) O UN ASTERISCO (*), EL REPORTE SERÁ RECHAZADO POR EL COMPILADOR DEL PDF.
- Asegúrate de cerrar siempre cada etiqueta <b> con su respectivo </b> en la misma línea.

Importante:
- No inventes umbrales.
- No inventes información sobre el usuario.
- No afirmes que el comportamiento es definitivamente scraping.
- Diferencia entre evidencia observada y una posible interpretación.
"""

    # Llamada a la API de Gemini para la sesión actual
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    
    # Guardamos el resultado en nuestro diccionario usando el ID de sesión como llave
    reportes_generados[s_id] = response.text
    print(f"¡Reporte para {s_id} generado con éxito!")

print("\n--- PROCESO TERMINADO ---")
print(f"Se han generado {len(reportes_generados)} reportes en la variable 'reportes_generados'.")


In [62]:
pip install -U google-genai

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------------------------- ------------ 0.8/1.1 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 4.1 MB/s eta 0:00:00
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.23.0
    Uninstalling google-genai-2.23.0:
      Successfully uninstalled google-genai-2.23.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install python-dotenv

  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak  # <-- Agregamos PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.units import cm
from xml.sax.saxutils import escape

# 1. Definimos un único nombre de archivo consolidado
filename = "reporte_sesiones_sospechosas_consolidado.pdf"

doc = SimpleDocTemplate(
    filename,
    pagesize=A4,
    rightMargin=2*cm,
    leftMargin=2*cm,
    topMargin=2*cm,
    bottomMargin=2*cm
)

styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    "ReportTitle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    spaceAfter=15
)

body_style = ParagraphStyle(
    "ReportBody",
    parent=styles["BodyText"],
    leading=16,
    spaceAfter=10
)

# Inicializamos la lista story única para todo el documento
story = []

# 2. Iteramos sobre cada sesión y su reporte generado por Gemini
# (Asegúrate de haber corrido el bucle de Gemini antes de ejecutar este código)
for index, (s_id, report_text) in enumerate(reportes_generados.items()):
    
    # Si no es la primera sesión, insertamos un salto de página obligatorio antes de la nueva sección
    if index > 0:
        story.append(PageBreak())
    
    # Estructura de cabecera para cada sesión individual
    story.append(Paragraph("<b>Reporte de Sesión Sospechosa</b>", title_style))
    story.append(Paragraph(f"<b>Session ID:</b> {escape(str(s_id))}", body_style))
    story.append(Spacer(1, 0.5 * cm))
    
    # Procesamos las líneas del reporte actual de la misma forma que antes
    for line in report_text.split("\n"):
        line_raw = line.strip()
        
        if line_raw:
            # Limpieza preventiva de Markdown residual
            line = line.replace("*", "").replace("`", '"')
            
            # Conservar tabulaciones iniciales
            leading_spaces = len(line) - len(line.lstrip(' '))
            nbsp_prefix = "&nbsp;" * leading_spaces
            
            formatted_text = f"{nbsp_prefix}{line_raw}"
            
            # Espaciado extra si la línea arranca como un apartado principal
            if line_raw.startswith(("1.", "2.", "3.", "4.", "5.", "6.")):
                story.append(Spacer(1, 0.4 * cm))
                
            story.append(Paragraph(formatted_text, body_style))

# 3. Compilamos el PDF final una sola vez con todas las páginas unificadas
doc.build(story)
print(f"PDF consolidado generado correctamente con {len(reportes_generados)} páginas base: {filename}")


PDF consolidado generado correctamente con 3 páginas base: reporte_sesiones_sospechosas_consolidado.pdf


: 